# Task 1: Marginal Distributions — All Scalar Variables, Globally

## Variable naming note

There are three naming layers in this project and they don't align:

| Layer | Example | Where used |
|---|---|---|
| `basin08` column | `ari_ix_sav` | PostgreSQL table — what we query |
| API key | `aridity` | JSON response from `/api/signature` |
| Schema key | `aridity_index` | `metadata/edops_codebook.tsv`, docs |

This notebook works at the **basin08 column** level (what we query directly) but labels outputs with the **API key** (what appears in the signature and is more readable). The codebook at `metadata/edops_codebook.tsv` is the authoritative mapping between all three layers.

**Temperature fields** (`tmp_dc_*`) are stored in basin08 as °C × 10. We divide by 10 before plotting.

**Scope**: implemented fields only — columns that are actually present in basin08 and currently returned by the API. Planned fields are excluded.

In [ ]:
import sys
sys.path.insert(0, '/Users/karlg/Documents/Repos/_cedop')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import skew
from scipy.stats import entropy as scipy_entropy

from scripts.shared.db_utils import db_connect

print('Imports OK')

## Variable definitions

Two lists:
- `SCALARS`: continuous/numeric fields — these get histograms and summary statistics.
- `CATEGORICALS`: integer-coded class fields — these get frequency counts and entropy.

`pnv_shares` (the compositional PNV field) is handled separately at the end.

Each scalar entry is `(basin08_column, api_key, scale_factor, units)`. Scale factor is 0.1 for temperature fields, 1.0 for everything else.

In [ ]:
# (basin08_col, api_key, scale_factor, units)
SCALARS = [
    # Band A — Terrain
    ('ele_mt_smn', 'elev_min',              1.0, 'm'),
    ('ele_mt_smx', 'elev_max',              1.0, 'm'),
    ('slp_dg_sav', 'slope_avg',             1.0, 'degrees'),
    ('slp_dg_uav', 'slope_upstream',        1.0, 'degrees'),
    ('sgr_dk_sav', 'stream_gradient',       1.0, 'm/km'),
    ('kar_pc_sse', 'karst',                 1.0, '%'),
    ('kar_pc_use', 'karst_upstream',        1.0, '%'),
    ('prm_pc_sse', 'permafrost_extent',     1.0, '%'),
    # Band B — Hydrology & soils
    ('dis_m3_pyr', 'discharge_yr',          1.0, 'm³/s'),
    ('dis_m3_pmn', 'discharge_min',         1.0, 'm³/s'),
    ('dis_m3_pmx', 'discharge_max',         1.0, 'm³/s'),
    ('run_mm_syr', 'runoff',                1.0, 'mm/yr'),
    ('gwt_cm_sav', 'gw_table_depth',        1.0, 'cm'),
    ('ria_ha_ssu', 'river_area',            1.0, 'ha'),
    ('ria_ha_usu', 'river_area_upstream',   1.0, 'ha'),
    ('wet_pc_sg1', 'wet_pct_grp1',          1.0, '%'),
    ('wet_pc_ug1', 'wet_pct_grp1_upstream', 1.0, '%'),
    ('wet_pc_sg2', 'wet_pct_grp2',          1.0, '%'),
    ('rev_mc_usu', 'reservoir_vol',         1.0, 'km³'),
    ('cly_pc_sav', 'pct_clay',              1.0, '%'),
    ('slt_pc_sav', 'pct_silt',              1.0, '%'),
    ('snd_pc_sav', 'pct_sand',              1.0, '%'),
    # Band C — Climate
    ('tmp_dc_syr', 'temp_yr',               0.1, '°C'),
    ('tmp_dc_uyr', 'temp_yr_upstream',      0.1, '°C'),
    ('tmp_dc_smn', 'temp_min',              0.1, '°C'),
    ('tmp_dc_smx', 'temp_max',              0.1, '°C'),
    ('pre_mm_syr', 'precip_yr',             1.0, 'mm/yr'),
    ('pre_mm_uyr', 'precip_yr_upstream',    1.0, 'mm/yr'),
    ('ari_ix_sav', 'aridity',               1.0, 'P/PET'),
    ('ari_ix_uav', 'aridity_upstream',      1.0, 'P/PET'),
    # Band D — Human
    ('crp_pc_sse', 'cropland_extent',           1.0, '%'),
    ('crp_pc_use', 'cropland_extent_upstream',  1.0, '%'),
    ('ppd_pk_sav', 'pop_density',               1.0, 'pk/km²'),
    ('hft_ix_s09', 'human_footprint_09',        1.0, 'index'),
    ('hft_ix_u09', 'human_footprint_09_upstream',1.0,'index'),
    ('gdp_ud_sav', 'gdp_avg',                   1.0, 'USD/km²'),
    ('hdi_ix_sav', 'human_dev_idx',             1.0, 'index'),
    # Band E / Coastality
    ('dist_sink',  'dist_sink',                 1.0, 'km'),
]

# Integer-coded categorical fields
# We use the integer column directly; lookup tables resolve to names but aren't needed for frequency/entropy
CATEGORICALS = [
    ('lit_cl_smj', 'lith_class',             'Lithology class'),
    ('clz_cl_smj', 'zone_id',                'Climate zone'),
    ('cls_cl_smj', 'strata_id',              'Climate stratum'),
    ('tbi_cl_smj', 'biome_id',               'Biome'),
    ('tec_cl_smj', 'eco_id',                 'Terrestrial ecoregion'),
    ('pnv_cl_smj', 'pnveg_id',               'PNV majority class'),
    ('wet_cl_smj', 'wetland_class',          'Wetland class'),
    ('fmh_cl_smj', 'freshwater_type',        'Freshwater habitat type'),
    ('fec_cl_smj', 'freshwater_ecoregion_name', 'Freshwater ecoregion'),
]

# PNV shares columns (compositional — handled separately)
PNV_SHARE_COLS = [f'pnv_pc_s{i:02d}' for i in range(1, 16)]

scalar_cols   = [s[0] for s in SCALARS]
cat_cols      = [c[0] for c in CATEGORICALS]
all_cols      = scalar_cols + cat_cols + PNV_SHARE_COLS

print(f'{len(scalar_cols)} scalar columns')
print(f'{len(cat_cols)} categorical columns')
print(f'{len(PNV_SHARE_COLS)} PNV share columns')

## Pull data from basin08

Single query: all 190k rows, all columns at once. Loads into a pandas DataFrame (~50–100MB in memory). We'll work from this DataFrame for the rest of the notebook — no further DB queries needed.

In [ ]:
conn = db_connect()

# Build SELECT — only request columns that exist; dist_sink may not yet be in basin08
# We'll verify column existence first
with conn.cursor() as cur:
    cur.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = 'basin08'
    """)
    existing = {row[0] for row in cur.fetchall()}

available     = [c for c in all_cols if c in existing]
not_available = [c for c in all_cols if c not in existing]

if not_available:
    print(f'Not in basin08 (skipping): {not_available}')

col_sql = ', '.join(f'"{c}"' for c in available)
df_raw = pd.read_sql(f'SELECT {col_sql} FROM public.basin08', conn)
conn.close()

print(f'Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns')
df_raw.head(3)

## Apply scale factors

Temperature columns are stored ×10 in basin08 — multiply by 0.1 to get °C. All other columns are already in their natural units.

In [ ]:
df = df_raw.copy()

for basin_col, api_key, scale, units in SCALARS:
    if basin_col in df.columns and scale != 1.0:
        df[basin_col] = df[basin_col] * scale
        print(f'  Scaled {basin_col} ({api_key}) × {scale}')

print('Scale factors applied.')

---
## Part 1: Scalar distributions

For each scalar variable, we want to know:
- **Shape**: Is the distribution spread out (informative) or piled up in one place (degenerate or near-degenerate)?
- **Center and spread**: Mean, median, standard deviation
- **Skewness**: How lopsided is it? Right-skewed means a long tail of high values (common for discharge, population).
- **Missingness**: % null (no data) and % zero (data, but zero value — different problem)

The summary table assembled here is the primary product of Task 1 for scalars.

In [ ]:
def scalar_summary(series, api_key, units):
    """Compute summary statistics for one scalar variable."""
    n_total = len(series)
    n_null  = series.isna().sum()
    valid   = series.dropna()
    n_zero  = (valid == 0).sum()

    if len(valid) == 0:
        return dict(api_key=api_key, units=units, n_valid=0,
                    pct_null=100.0, pct_zero=None,
                    mean=None, median=None, std=None, skewness=None,
                    p05=None, p95=None, classification='ALL NULL')

    pct_null = 100 * n_null / n_total
    pct_zero = 100 * n_zero / len(valid)
    sk       = skew(valid)

    # Classification heuristic — we'll revisit after seeing the histograms
    if pct_null > 80 or pct_zero > 80:
        classification = 'degenerate'
    elif abs(sk) > 5 or valid.std() / (abs(valid.mean()) + 1e-9) > 10:
        classification = 'informative (heavy-tailed)'
    else:
        classification = 'informative'

    return dict(
        api_key        = api_key,
        units          = units,
        n_valid        = len(valid),
        pct_null       = round(pct_null, 1),
        pct_zero       = round(pct_zero, 1),
        mean           = round(valid.mean(), 3),
        median         = round(valid.median(), 3),
        std            = round(valid.std(), 3),
        skewness       = round(sk, 2),
        p05            = round(valid.quantile(0.05), 3),
        p95            = round(valid.quantile(0.95), 3),
        classification = classification,
    )

rows = []
for basin_col, api_key, scale, units in SCALARS:
    if basin_col in df.columns:
        rows.append(scalar_summary(df[basin_col], api_key, units))

summary = pd.DataFrame(rows)
summary

### Histograms — all scalar variables

One histogram per variable. X-axis clipped at the 1st–99th percentile so extreme outliers don't compress the picture into a thin spike. The actual min/max are in the summary table above.

**What to look for**:
- A spike at zero with most of the mass there → likely degenerate for this purpose
- A single bell-shaped peak → unimodal, probably a geographic gradient
- Two or more peaks → multimodal, potentially more interesting (different environmental regimes)
- Long right tail (most basins are low, a few are very high) → right-skewed; common for discharge, population

In [ ]:
available_scalars = [(bc, ak, sf, u) for bc, ak, sf, u in SCALARS if bc in df.columns]
n_plots = len(available_scalars)
ncols = 4
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3.5))
axes = axes.flatten()

for i, (basin_col, api_key, scale, units) in enumerate(available_scalars):
    ax  = axes[i]
    col = df[basin_col].dropna()

    # Clip to 1–99th percentile for display
    lo, hi = col.quantile(0.01), col.quantile(0.99)
    clipped = col[(col >= lo) & (col <= hi)]

    ax.hist(clipped, bins=60, color='steelblue', alpha=0.8, edgecolor='none')
    ax.set_title(f'{api_key}', fontsize=9, fontweight='bold')
    ax.set_xlabel(units, fontsize=7)
    ax.set_ylabel('basins', fontsize=7)
    ax.tick_params(labelsize=7)

    # Annotate with % null and skew
    pct_null = 100 * df[basin_col].isna().sum() / len(df)
    sk_val   = skew(col)
    ax.text(0.97, 0.95, f'null={pct_null:.1f}%  skew={sk_val:.1f}',
            transform=ax.transAxes, fontsize=6,
            ha='right', va='top', color='#555')

# Hide unused axes
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('EDOPS — L8 Scalar Marginal Distributions (190k basins, clipped to p1–p99)', 
             fontsize=12, y=1.01)
plt.tight_layout()

import os
os.makedirs('/Users/karlg/Documents/Repos/_cedop/output/edop/explore', exist_ok=True)
fig.savefig('/Users/karlg/Documents/Repos/_cedop/output/edop/explore/01_scalar_histograms.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved to output/edop/explore/01_scalar_histograms.png')

### Summary table — sorted by classification

Degenerate variables appear first so they're easy to spot. The classification is a first-pass heuristic — override it by hand in the cell below after reviewing the histograms.

In [ ]:
summary_sorted = summary.sort_values('classification')
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
summary_sorted

In [ ]:
# Save summary table to CSV
summary_sorted.to_csv(
    '/Users/karlg/Documents/Repos/_cedop/output/edop/explore/01_scalar_summary.csv',
    index=False
)
print('Saved to output/edop/explore/01_scalar_summary.csv')

### Manual classification overrides

After reviewing the histograms, update any classifications that the heuristic got wrong. Edit the dict below and re-run.

In [ ]:
# Add manual overrides here, keyed by api_key
# Example: overrides = {'reservoir_vol': 'degenerate', 'karst': 'informative'}
overrides = {}

if overrides:
    summary['classification'] = summary.apply(
        lambda r: overrides.get(r['api_key'], r['classification']), axis=1
    )
    print('Overrides applied:', overrides)
else:
    print('No overrides set.')

---
## Part 2: Categorical distributions

For categorical fields, we don't compute means — we count how many basins fall into each class. The questions are:
- Is one class dominant (e.g., 80% of basins in one biome)? That's low entropy — the variable is less useful for distinguishing basins.
- Are classes roughly evenly distributed? That's high entropy — more signal.
- Are there rare classes (e.g., a lithology type in <0.1% of basins)? These won't show up meaningfully in any downstream analysis.

In [ ]:
def categorical_summary(series, api_key, label):
    """Frequency count and entropy for one categorical variable."""
    counts   = series.value_counts(dropna=True)
    n_total  = len(series)
    n_null   = series.isna().sum()
    probs    = counts / counts.sum()
    H        = scipy_entropy(probs, base=2)  # bits
    H_max    = np.log2(len(counts)) if len(counts) > 1 else 1
    return dict(
        api_key      = api_key,
        label        = label,
        n_classes    = len(counts),
        pct_null     = round(100 * n_null / n_total, 1),
        entropy_bits = round(H, 2),
        entropy_norm = round(H / H_max, 3) if H_max > 0 else None,  # 0=one class dominates, 1=perfectly even
        top_class_id = counts.index[0],
        top_class_pct= round(100 * counts.iloc[0] / counts.sum(), 1),
    )

cat_rows = []
for basin_col, api_key, label in CATEGORICALS:
    if basin_col in df.columns:
        cat_rows.append(categorical_summary(df[basin_col], api_key, label))

cat_summary = pd.DataFrame(cat_rows).sort_values('entropy_norm', ascending=False)
cat_summary

In [ ]:
# Bar chart per categorical variable
available_cats = [(bc, ak, lbl) for bc, ak, lbl in CATEGORICALS if bc in df.columns]
fig, axes = plt.subplots(len(available_cats), 1,
                          figsize=(14, 4 * len(available_cats)))
if len(available_cats) == 1:
    axes = [axes]

for ax, (basin_col, api_key, label) in zip(axes, available_cats):
    counts = df[basin_col].value_counts(dropna=True).head(30)  # top 30 classes
    ax.bar(range(len(counts)), counts.values, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'{label}  ({api_key})  — top {len(counts)} classes', fontsize=9)
    ax.set_ylabel('basins', fontsize=8)

plt.tight_layout()
fig.savefig('/Users/karlg/Documents/Repos/_cedop/output/edop/explore/01_categorical_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved to output/edop/explore/01_categorical_distributions.png')

In [ ]:
cat_summary.to_csv(
    '/Users/karlg/Documents/Repos/_cedop/output/edop/explore/01_categorical_summary.csv',
    index=False
)
print('Saved to output/edop/explore/01_categorical_summary.csv')

---
## Part 3: PNV shares — compositional analysis

`pnv_pc_s01` through `pnv_pc_s15` are the shares of each Potential Natural Vegetation class within a basin. They sum to approximately 100 per basin. This is a compositional variable — the question isn't the distribution of any one class, but how much variation there is within each basin across classes.

We measure diversity per basin using Shannon entropy (same formula as above, applied per row rather than per column). A basin with one dominant PNV class gets low entropy; a basin evenly split across many classes gets high entropy.

In [ ]:
pnv_cols_available = [c for c in PNV_SHARE_COLS if c in df.columns]
print(f'{len(pnv_cols_available)} PNV share columns available')

if pnv_cols_available:
    pnv_df = df[pnv_cols_available].fillna(0)

    # Normalize each row to proportions (shares should already sum to ~100)
    row_sums = pnv_df.sum(axis=1).replace(0, np.nan)
    pnv_props = pnv_df.div(row_sums, axis=0)

    # Shannon entropy per basin (base 2, in bits)
    # Replace 0 with a tiny value to avoid log(0)
    pnv_entropy = pnv_props.apply(
        lambda row: scipy_entropy(row[row > 0], base=2), axis=1
    )

    print(f'PNV diversity (Shannon entropy, bits) across {len(pnv_entropy):,} basins:')
    print(pnv_entropy.describe().round(3))

    # How many basins have one dominant class at >95%?
    single_dominant = (pnv_props.max(axis=1) > 0.95).sum()
    print(f'\nBasins with one PNV class >95%: {single_dominant:,} ({100*single_dominant/len(pnv_df):.1f}%)')
    print(f'Basins with mixed PNV (<95% dominant): {len(pnv_df)-single_dominant:,}')

In [ ]:
if pnv_cols_available:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.hist(pnv_entropy.dropna(), bins=50, color='steelblue', alpha=0.8)
    ax1.set_xlabel('Shannon entropy (bits)')
    ax1.set_ylabel('basins')
    ax1.set_title('PNV diversity per basin')

    dominant_share = pnv_props.max(axis=1)
    ax2.hist(dominant_share.dropna(), bins=50, color='darkorange', alpha=0.8)
    ax2.set_xlabel('Share of most common PNV class')
    ax2.set_ylabel('basins')
    ax2.set_title('Dominant PNV class share per basin')
    ax2.axvline(0.95, color='red', linestyle='--', linewidth=1, label='>95% threshold')
    ax2.legend(fontsize=8)

    plt.suptitle('PNV Shares — Compositional Diversity', fontsize=11)
    plt.tight_layout()
    fig.savefig('/Users/karlg/Documents/Repos/_cedop/output/edop/explore/01_pnv_diversity.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to output/edop/explore/01_pnv_diversity.png')

---
## Findings

Record observations here as you review the outputs above. These will be transferred to `logs/exploration_log.md`.

Template:
```
**Variable**: xxx  
**Finding**: xxx  
**Implication**: xxx
```

*(add findings here)*